In [1]:
# Import Required Libraries
import sys
import os
import json
import folium
from ipywidgets import VBox, HBox, Checkbox, Output, Label, HTML, Button
from IPython.display import display, clear_output
import geopandas as gpd

# Add path for custom modules
sys.path.insert(0, '/home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas')

from maritime_layers.utils.map_layers import (
    TerritorialSeaLayer,
    ContiguousZoneLayer,
    EEZLayer,
    FAOFishingAreaLayer
)

# Base path
base_path = '/home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas'

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Initialize Base Map (Single Map Instance)
print("\nInitializing single interactive map...")

# Create a persistent base map
base_map = folium.Map(
    location=[0, 0],
    zoom_start=2,
    tiles='OpenStreetMap'
)

# Add layer control
folium.LayerControl().add_to(base_map)

# Add legend
legend_html = '''
<div style="position: fixed; 
        bottom: 50px; right: 50px; width: 320px; height: auto; 
        background-color: white; border:3px solid #333; z-index:9999; 
        font-size:13px; padding: 15px; border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
<h3 style="margin-top: 0; color: #333;">Maritime Boundaries</h3>
<hr style="margin: 10px 0;">
<p style="margin: 8px 0; font-size: 11px;"><b style="color: #c92a2a;">■</b> 12NM Territorial Sea</p>
<p style="margin: 8px 0; font-size: 11px;"><b style="color: #0051ba;">■</b> 24NM Contiguous Zone</p>
<p style="margin: 8px 0; font-size: 11px;"><b style="color: #1b5e20;">■</b> Exclusive Economic Zone</p>
<p style="margin: 8px 0; font-size: 11px;"><b style="color: #ff7f0e;">■</b> FAO Fishing Areas</p>
<hr style="margin: 10px 0;">
<p style="margin: 5px 0; font-size: 10px; color: #666;">Click on any zone for details</p>
</div>
'''

base_map.get_root().html.add_child(folium.Element(legend_html))

print("✓ Base map created")


Initializing single interactive map...
✓ Base map created


In [3]:
# Load All Maritime Boundary Data
print("\nLoading maritime boundary data...")

# Initialize layers
layers = {}

# 12NM Territorial Sea
print("  Loading 12NM Territorial Sea...")
territorial_sea = TerritorialSeaLayer()
territorial_sea_path = os.path.join(
    base_path, 'World_Boundaries', 'World_12NM_v4_20231025_gpkg', 'eez_12nm_v4.gpkg'
)
territorial_sea.load_data(territorial_sea_path)
layers['12NM Territorial Sea'] = territorial_sea

# 24NM Contiguous Zone
print("  Loading 24NM Contiguous Zone...")
contiguous_zone = ContiguousZoneLayer()
contiguous_zone_path = os.path.join(
    base_path, 'World_Boundaries', 'World_24NM_v4_20231025_gpkg', 'eez_24nm_v4.gpkg'
)
contiguous_zone.load_data(contiguous_zone_path)
layers['24NM Contiguous Zone'] = contiguous_zone

# EEZ
print("  Loading Exclusive Economic Zone...")
eez = EEZLayer()
eez_path = os.path.join(
    base_path, 'World_EEZ_v12_20231025_gpkg', 'eez_v12.gpkg'
)
eez.load_data(eez_path)
layers['Exclusive Economic Zone (EEZ)'] = eez

# FAO Fishing Areas
print("  Loading FAO Fishing Areas...")
fao_areas = FAOFishingAreaLayer()
fao_path = os.path.join(base_path, 'FAO_AREAS_ERASE.json')
fao_areas.load_data(fao_path)
layers['FAO Fishing Areas'] = fao_areas

print("\n✓ All data loaded successfully")


Loading maritime boundary data...
  Loading 12NM Territorial Sea...
Loading 12NM Territorial Sea from /home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/World_Boundaries/World_12NM_v4_20231025_gpkg/eez_12nm_v4.gpkg...
✓ Loaded 230 territorial sea zones
  Loading 24NM Contiguous Zone...
Loading 24NM Contiguous Zone from /home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/World_Boundaries/World_24NM_v4_20231025_gpkg/eez_24nm_v4.gpkg...
✓ Loaded 220 contiguous zones
  Loading Exclusive Economic Zone...
Loading EEZ from /home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/World_EEZ_v12_20231025_gpkg/eez_v12.gpkg...
✓ Loaded 285 EEZ zones
  Loading FAO Fishing Areas...
Loading FAO Fishing Areas from /home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/FAO_AREAS_ERASE.json...
✓ Loaded 370 FAO fishing areas

✓ All data loaded successfully


In [4]:
# Create Feature Groups for Each Layer
print("\nCreating feature groups for each layer...")

feature_groups = {}

# Create feature group for 12NM Territorial Sea
fg_12nm = folium.FeatureGroup(name='12NM Territorial Sea', show=False)
for feature in layers['12NM Territorial Sea'].geojson_data['features']:
    folium.GeoJson(
        feature,
        style_function=lambda x: {
            'fillColor': '#ff6b6b',
            'color': '#c92a2a',
            'weight': 1,
            'opacity': 0.7,
            'fillOpacity': 0.5
        },
        popup=folium.Popup(
            f"<b>{feature['properties'].get('GEONAME', 'Unknown')}</b><br>"
            f"Territory: {feature['properties'].get('TERRITORY1', 'N/A')}<br>"
            f"Area: {feature['properties'].get('AREA_KM2', 'N/A'):,} km²",
            max_width=300
        )
    ).add_to(fg_12nm)
fg_12nm.add_to(base_map)
feature_groups['12NM Territorial Sea'] = fg_12nm
print("  ✓ 12NM Territorial Sea")

# Create feature group for 24NM Contiguous Zone
fg_24nm = folium.FeatureGroup(name='24NM Contiguous Zone', show=False)
for feature in layers['24NM Contiguous Zone'].geojson_data['features']:
    folium.GeoJson(
        feature,
        style_function=lambda x: {
            'fillColor': '#1f77b4',
            'color': '#0051ba',
            'weight': 1,
            'opacity': 0.7,
            'fillOpacity': 0.5
        },
        popup=folium.Popup(
            f"<b>{feature['properties'].get('GEONAME', 'Unknown')}</b><br>"
            f"Territory: {feature['properties'].get('TERRITORY1', 'N/A')}<br>"
            f"Area: {feature['properties'].get('AREA_KM2', 'N/A'):,} km²",
            max_width=300
        )
    ).add_to(fg_24nm)
fg_24nm.add_to(base_map)
feature_groups['24NM Contiguous Zone'] = fg_24nm
print("  ✓ 24NM Contiguous Zone")

# Create feature group for EEZ
fg_eez = folium.FeatureGroup(name='Exclusive Economic Zone (EEZ)', show=False)
for feature in layers['Exclusive Economic Zone (EEZ)'].geojson_data['features']:
    folium.GeoJson(
        feature,
        style_function=lambda x: {
            'fillColor': '#2ca02c',
            'color': '#1b5e20',
            'weight': 1,
            'opacity': 0.7,
            'fillOpacity': 0.4
        },
        popup=folium.Popup(
            f"<b>{feature['properties'].get('GEONAME', 'Unknown')}</b><br>"
            f"Territory: {feature['properties'].get('TERRITORY1', 'N/A')}<br>"
            f"Area: {feature['properties'].get('AREA_KM2', 'N/A'):,} km²",
            max_width=300
        )
    ).add_to(fg_eez)
fg_eez.add_to(base_map)
feature_groups['Exclusive Economic Zone (EEZ)'] = fg_eez
print("  ✓ Exclusive Economic Zone (EEZ)")

# Create feature group for FAO Fishing Areas
fg_fao = folium.FeatureGroup(name='FAO Fishing Areas', show=False)
ocean_colors = {
    'Arctic': '#1f77b4',
    'Atlantic': '#ff7f0e',
    'Pacific': '#2ca02c',
    'Indian': '#d62728',
    'Southern': '#9467bd',
    'Mediterranean': '#8c564b'
}

for feature in layers['FAO Fishing Areas'].geojson_data['features']:
    ocean = feature['properties'].get('OCEAN', 'Arctic')
    color = ocean_colors.get(ocean, '#cccccc')
    
    folium.GeoJson(
        feature,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': color,
            'weight': 1,
            'opacity': 0.7,
            'fillOpacity': 0.3
        },
        popup=folium.Popup(
            f"<b>{feature['properties'].get('F_NAME', 'Unknown')}</b><br>"
            f"Code: {feature['properties'].get('F_CODE', 'N/A')}<br>"
            f"Ocean: {feature['properties'].get('OCEAN', 'N/A')}<br>"
            f"Status: {feature['properties'].get('F_STATUS', 'N/A')}",
            max_width=300
        )
    ).add_to(fg_fao)
fg_fao.add_to(base_map)
feature_groups['FAO Fishing Areas'] = fg_fao
print("  ✓ FAO Fishing Areas")

print("\n✓ All feature groups created")


Creating feature groups for each layer...
  ✓ 12NM Territorial Sea
  ✓ 24NM Contiguous Zone
  ✓ Exclusive Economic Zone (EEZ)
  ✓ FAO Fishing Areas

✓ All feature groups created


In [5]:
# Create Interactive Control Panel with Checkboxes
print("\nCreating interactive control panel...")

# Output widget for the map
map_output = Output()
info_output = Output()

# Create checkboxes for each layer
cb_12nm = Checkbox(
    value=False,
    description='12NM Territorial Sea',
    indent=False,
    style={'description_width': '200px'}
)

cb_24nm = Checkbox(
    value=False,
    description='24NM Contiguous Zone',
    indent=False,
    style={'description_width': '200px'}
)

cb_eez = Checkbox(
    value=False,
    description='Exclusive Economic Zone (EEZ)',
    indent=False,
    style={'description_width': '200px'}
)

cb_fao = Checkbox(
    value=False,
    description='FAO Fishing Areas',
    indent=False,
    style={'description_width': '200px'}
)

# Store checkbox references
checkboxes = {
    '12NM Territorial Sea': cb_12nm,
    '24NM Contiguous Zone': cb_24nm,
    'Exclusive Economic Zone (EEZ)': cb_eez,
    'FAO Fishing Areas': cb_fao
}

print("✓ Checkboxes created")


Creating interactive control panel...
✓ Checkboxes created


In [6]:
# Define Callback Function to Update Layer Visibility
def update_layer_visibility(change=None):
    """Update layer visibility based on checkbox states"""
    
    with info_output:
        clear_output(wait=True)
        
        # Check which layers are selected
        selected_count = sum([cb.value for cb in checkboxes.values()])
        
        if selected_count == 0:
            print("ℹ️  No layers selected. Select one or more layers to display.")
        else:
            selected_layers = [name for name, cb in checkboxes.items() if cb.value]
            print(f"✓ Displaying {selected_count} layer(s):")
            for layer_name in selected_layers:
                print(f"  • {layer_name}")
        
        # Update feature group visibility
        for layer_name, fg in feature_groups.items():
            if checkboxes[layer_name].value:
                fg.show = True
            else:
                fg.show = False
        
        # Refresh the map display
        with map_output:
            clear_output(wait=True)
            display(base_map)

# Attach callbacks to checkboxes
for cb in checkboxes.values():
    cb.observe(update_layer_visibility, names='value')

print("✓ Callbacks attached to checkboxes")

✓ Callbacks attached to checkboxes


In [ ]:
# Display the Interactive Interface
print("\n" + "="*70)
print("INTERACTIVE MARITIME BOUNDARIES VISUALIZATION")
print("="*70)

# Create header
header = HTML("""
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 8px; margin-bottom: 20px; color: white;">
    <h2 style="margin: 0; font-size: 24px;">🗺️ Maritime Boundaries Explorer</h2>
    <p style="margin: 10px 0 0 0; font-size: 14px; opacity: 0.95;">
        Select one or more maritime boundary layers to display on the map. 
        Click on any zone for detailed information.
    </p>
</div>
""")

# Create control section
control_label = HTML("<h3 style='margin: 0 0 15px 0; color: #333;'>Select Layers to Display:</h3>")

# Group checkboxes
checkbox_group = VBox([
    cb_12nm,
    cb_24nm,
    cb_eez,
    cb_fao
])

# Create control panel
control_panel = VBox([
    header,
    control_label,
    checkbox_group,
    HTML("<hr style='margin: 20px 0;'>"),
    info_output
])

# Display everything
display(control_panel)
display(HTML("<h3 style='margin-top: 20px; margin-bottom: 10px;'>Map:</h3>"))
display(map_output)

# Display initial map
update_layer_visibility()


INTERACTIVE MARITIME BOUNDARIES VISUALIZATION


HTML(value="<h3 style='margin-top: 20px; margin-bottom: 10px;'>Map:</h3>")

Output()

In [ ]:
# Export Current Map
output_path = os.path.join(base_path, 'maritime_interactive_single_map.html')
base_map.save(output_path)
print(f"\n✓ Interactive map saved to:")
print(f"  {output_path}")
print(f"\nYou can open this file in any web browser to explore the maritime boundaries.")

## How to Use

1. **Toggle Layers**: Check/uncheck the boxes above to show/hide different maritime boundary layers
2. **View Details**: Click on any zone on the map to see detailed information
3. **Zoom & Pan**: Use the map controls to zoom in/out and pan around
4. **Layer Legend**: Check the legend on the map (bottom-right) for color information

## Layer Information

### 12NM Territorial Sea (Red)
- Maritime zone extending 12 nautical miles from a country's baseline
- Country exercises full sovereignty over this zone
- 230 territorial sea zones worldwide

### 24NM Contiguous Zone (Blue)
- Maritime zone extending 24 nautical miles from baseline
- Used for enforcing laws on immigration, customs, and pollution
- 220 contiguous zones worldwide

### Exclusive Economic Zone (EEZ) (Green)
- Maritime zone extending 200 nautical miles from baseline
- Country has exclusive rights to explore and exploit natural resources
- 285 EEZ zones worldwide
- Total area: 140.8 million km²

### FAO Fishing Areas (Multi-color)
- Fishing zones defined by the Food and Agriculture Organization (UN)
- Color-coded by ocean: Atlantic, Pacific, Indian, Arctic, Mediterranean
- 370 fishing areas worldwide